In [ ]:
# Day 29 - Capstone: Integrated RAG + Memory Assistant

!pip install -q langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu

from langchain_huggingface import HuggingFacePipeline
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

import torch
from transformers import pipeline

print("✅ All tools loaded!")

In [ ]:
# 1. Load LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=250,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# 2. Knowledge Base + RAG Setup
knowledge = """
Ethiopia is one of the oldest nations in the world.
Addis Ababa is the capital and diplomatic hub of Africa.
Injera is the national dish.
Coffee originated in Ethiopia.
Lalibela is famous for its rock-hewn churches.
"""

# Simple RAG
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
texts = text_splitter.create_documents([knowledge])
vectorstore = FAISS.from_documents(texts, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
# 3. Memory + RAG Combined
memory = ConversationBufferMemory()

def integrated_assistant(query):
    # Retrieve context
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])
    
    # Use memory + context
    full_input = f"Context: {context}\nUser: {query}"
    response = llm.invoke(full_input)
    print("AI:", response.strip())
    return response

# Test
integrated_assistant("What is special about Ethiopia?")
integrated_assistant("Tell me more about its food.")